# Skyline Matching Evaluation (Colab, Resumable)

Assumes:
- Repo cloned at `/content/SkylineGeolocation`
- Drive has: `MyDrive/skyline_db.parquet`, `MyDrive/synthetic_dataset`
- Drive has: `MyDrive/sky_segmentation_unet_model.pth`

**Resumable**: runs `scripts/verify_artifacts.py` first. Skips any step whose
output already exists. Saves eval results to Drive after completion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -qq libgl1-mesa-glx libglib2.0-0 2>/dev/null
!pip install -q fastdtw pyarrow geopy pyprojroot segmentation-models-pytorch timm albumentations

In [ ]:
import os, sys, json
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
DRIVE = Path('/content/drive/MyDrive')
STATE = DRIVE / 'pipeline_state.json'
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from scripts.resume import ResumeManifest
m = ResumeManifest(STATE)
print(m.summary() if m.state else 'Fresh start')

In [ ]:
# Link Drive files into repo (idempotent)
links = [
    (DRIVE / 'skyline_db.parquet',
     REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'),
    (DRIVE / 'synthetic_dataset',
     REPO / 'data/synthetic_dataset'),
    (DRIVE / 'sky_segmentation_unet_model.pth',
     REPO / 'data/sky_segmentation_unet_model.pth'),
    (DRIVE / 'projectdata/dem.tif',
     REPO / 'data/digital_elevation_model/dem_30m.tif'),
]

for src, dst in links:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(src, dst)
        print(f'Linked {src.name}')
    elif dst.exists():
        print(f'{dst.name}: already linked')
    else:
        print(f'MISSING: {src}')

In [ ]:
# Verify DB integrity (parquet header/footer check)
db_path = REPO / 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'
if db_path.exists():
    size = db_path.stat().st_size
    if size < 900_000_000:
        raise SystemExit(f'BAD: DB size {size} bytes < 900 MB. Re-upload.')
    with open(db_path, 'rb') as f:
        f.seek(0); hdr = f.read(4)
        f.seek(-8, 2); ftr = f.read()
    if hdr != b'PAR1' or ftr[-4:] != b'PAR1':
        raise SystemExit('BAD: DB parquet magic bytes missing')
    print(f'DB OK: {size / 1e6:.1f} MB, PAR1 headers intact')
else:
    raise SystemExit('MISSING: DB parquet not found')

In [ ]:
# Check ground_truth.json exists (synthetic_dataset lacks it)
gt = REPO / 'data/synthetic_dataset/ground_truth.json'
if not gt.exists():
    print(f'MISSING: {gt}')
    print('Upload ground_truth.json to Drive/MyDrive/synthetic_dataset/ and re-run')
else:
    import json
    data = json.loads(gt.read_text())
    print(f'Ground truth: {len(data)} samples')
    m.mark('ground_truth', n_samples=len(data))
    m.save()

In [ ]:
# Check predicted_masks
masks_dir = REPO / 'data/synthetic_dataset/predicted_masks'
if not masks_dir.exists() or len(list(masks_dir.glob('*.png'))) < 300:
    print(f'predicted_masks incomplete. Run segment_synthetic.ipynb first or upload masks.')
    n = len(list(masks_dir.glob('*.png'))) if masks_dir.exists() else 0
    print(f'Have {n}/300 masks')
else:
    print(f'predicted_masks OK: {len(list(masks_dir.glob("*.png")))} files')
    m.mark('predicted_masks', n_files=300)
    m.save()

In [ ]:
# Skip eval if not resumable
gt = REPO / 'data/synthetic_dataset/ground_truth.json'
masks_dir = REPO / 'data/synthetic_dataset/predicted_masks'
if not gt.exists() or not masks_dir.exists() or len(list(masks_dir.glob('*.png'))) < 300:
    raise SystemExit('Cannot run eval: missing ground_truth or predicted_masks')

In [ ]:
import gc
import pandas as pd
import pyarrow.parquet as pq

meta = pd.read_parquet('notebooks/02_SkylineDatabase/output/skyline_db.parquet',
                       columns=['lon', 'lat', 'elevation_m'])
del meta; gc.collect()
print('DB metadata loaded', flush=True)

pf = pq.ParquetFile('notebooks/02_SkylineDatabase/output/skyline_db.parquet')
first = next(pf.iter_batches(batch_size=1, columns=['raw_horizon_deg']))
bin_deg = 360.0 / len(first.to_pandas()['raw_horizon_deg'].iloc[0])
print(f'bin_deg={bin_deg}', flush=True)

In [ ]:
results_path = REPO / 'notebooks/05_SkylineMatching/output/eval_results.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)

if m.done('eval') and results_path.exists():
    df = pd.read_csv(results_path)
    print(f'Loaded cached results: {len(df)} rows')
    summary = {'cached': True}
else:
    from src.evaluation import run_evaluation
    df, summary = run_evaluation(
        ground_truth_path='data/synthetic_dataset/ground_truth.json',
        db_path='notebooks/02_SkylineDatabase/output/skyline_db.parquet',
        masks_dir='data/synthetic_dataset/predicted_masks',
        use_altimeter=True,
        use_compass=True,
        limit=0,
        sample_batch_size=8,
        top_k=30,
        dtw_window=15,
        correct_dist_m=500.0,
        chunk_rows=4000,
        spatial_stride=5,
    )
    if len(df) > 0:
        df.to_csv(results_path, index=False)
        df.to_csv(DRIVE / 'eval_results.csv', index=False)
        m.mark('eval', n_samples=len(df), summary=summary)
        m.save()
        print(f'Saved {len(df)} rows')

print(json.dumps(summary, indent=2))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(df) > 0:
    errors = df['error_m']
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(errors, bins=50, color='steelblue', edgecolor='white')
    plt.axvline(500, color='red', ls='--', label='500m')
    plt.xlabel('Error (m)'); plt.ylabel('Count')
    plt.title(f'Top-1 Errors (n={len(errors)})'); plt.legend()

    plt.subplot(1, 2, 2)
    for dist, label in [(100, '100m'), (500, '500m'), (1000, '1km'), (5000, '5km')]:
        plt.bar(label, (errors <= dist).mean() * 100, color='seagreen')
    plt.ylabel('Top-1 Accuracy (%)')
    plt.title('Accuracy at Distance Thresholds')
    plt.tight_layout()
    png = REPO / 'eval_results.png'
    plt.savefig(png, dpi=150)
    plt.savefig(DRIVE / 'eval_results.png', dpi=150)
    plt.show()
    print(f'Median: {errors.median():.0f}m')
    print(f'Top-1@500m: {(errors <= 500).mean()*100:.1f}%')
    print(f'Top-5@500m: {df["top5_ok"].mean()*100:.1f}%')